# BinaryMatchboxNet KWS — Remote GPU (WSL / RTX 5070 Ti)

Colab 부트스트랩을 원격 GPU용으로 옮긴 버전. 위에서 아래로 셀을 실행해.

**Colab과 다른 점 (중요):**
- 이 원격 WSL 디스크는 **영속적**이야 — Colab처럼 세션 끊긴다고 날아가지 않아.
  그래서 `git reset --hard`, Google Drive 마운트, 데이터 재다운로드가 **전부 불필요**.
- 코드를 VSCode에서 **원격에 직접 편집**할 수 있어. `%autoreload`가 `.py` 변경을
  재시작 없이 반영해. (Colab처럼 로컬편집→push→pull 안 해도 됨.)
- torch는 이미 **2.11+cu128 (Blackwell)** 로 venv에 깔아뒀어. 아래 의존성 설치는
  torch를 **건드리지 않아** (requirements-colab.txt가 torch를 일부러 뺌).

커널은 우상단에서 **`.venv/bin/python`** 을 선택했는지 확인해.


In [ ]:
# --- 1. setup: run from repo root, enable autoreload ---------------------
# 원격 repo는 영속적이라 VSCode에서 직접 수정 가능. autoreload가 .py 변경을
# 런타임 재시작 없이 반영한다 (수정 후 import 셀 다시 돌리면 최신 코드 반영).
%load_ext autoreload
%autoreload 2

import os, sys
# 노트북 작업 디렉터리를 repo 루트로 맞춘다 (data/models/train import가 되도록).
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
REPO = os.getcwd()
print('repo root :', REPO)
print('python    :', sys.executable)   # .../.venv/bin/python 이어야 함

# 최신 코드 당겨오기 (fast-forward만; 로컬 편집이 있으면 건드리지 않음).
# 원격에서 직접 편집 중이면 아래 pull은 주석 처리한 채로 둬.
# !git pull --ff-only
# --no-pager 필수: VSCode의 !셸은 pty에 붙어서 git이 페이저(less)를 띄우고
# 입력을 기다리며 멈춘다. --no-pager 로 페이저를 꺼서 그 hang 을 방지.
!git --no-pager log -1 --oneline

In [ ]:
# --- 2. dependencies -----------------------------------------------------
# requirements-colab.txt 는 torch/torchaudio 를 일부러 뺀 파일(pyyaml/soundfile/
# pytest 등만). 우리는 torch 2.11+cu128 을 이미 깔았으니 이걸로 설치해야
# torch 가 다운그레이드되지 않는다.
!{sys.executable} -m pip install -q -r requirements-colab.txt

# soundfile 이 libsndfile 시스템 라이브러리를 요구할 수 있음. 나중에
# 'libsndfile' 관련 에러가 나면 원격 터미널에서:  sudo apt install -y libsndfile1

In [ ]:
# --- 3. environment check ------------------------------------------------
import torch, torchaudio
print('torch      ', torch.__version__)
print('torchaudio ', torchaudio.__version__)
print('cuda       ', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(CPU only)')
print('capability ', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else '-')

# Blackwell(sm_120) 커널이 실제로 도는지 스모크 테스트
if torch.cuda.is_available():
    x = torch.randn(2000, 2000, device='cuda')
    print('matmul ok  ', float((x @ x).sum()))
else:
    print('\n[!] GPU 미인식. nvidia-smi / 드라이버 / torch cu128 설치 확인 필요.')

In [ ]:
# --- 4. unit tests (binary ops, AFE, model, config guardrails) -----------
!{sys.executable} -m pytest -q

In [ ]:
# --- 5. what will actually be built --------------------------------------
!{sys.executable} experiments/inspect_model.py configs/base.yaml

In [ ]:
# --- 6. checkpoints -------------------------------------------------------
# Google Drive 없음. WSL 디스크가 영속적이라 runs/ 는 그냥 repo 안에 남는다
# (gitignore 처리됨). 마운트/심링크 할 것 없음.
os.makedirs('runs', exist_ok=True)
!ls -la runs/ 2>/dev/null || echo '(runs/ is empty)'

---
## Speech Commands v2

데이터셋(~2.3 GB)은 영속 디스크의 `~/datasets/speech_commands_v2` 에 **한 번만**
내려받는다 (repo 안 아님, 매 세션 재다운로드 아님).


In [ ]:
# --- 7. dataset (한 번 받으면 WSL 디스크에 영속) -------------------------
from data.speech_commands import ensure_dataset, build_dataloaders, class_names
from train.config import load_config
import collections

DATA_ROOT = os.path.expanduser('~/datasets/speech_commands_v2')
ensure_dataset(DATA_ROOT)          # Drive 캐시 tar 불필요 — 디스크가 영속적

cfg = load_config('configs/base.yaml')
cfg.data.root = DATA_ROOT
train_loader, val_loader, test_loader = build_dataloaders(
    cfg.data, batch_size=cfg.train.batch_size, sample_rate=cfg.afe.sample_rate)

print('classes:', class_names())
for name, dl in [('train', train_loader), ('val', val_loader), ('test', test_loader)]:
    print(f'{name:5s}: {len(dl.dataset):>6} clips, {len(dl)} batches')
waves, labels = next(iter(train_loader))
print('batch', tuple(waves.shape), 'label hist',
      dict(sorted(collections.Counter(labels.tolist()).items())))

In [ ]:
# --- 8a. single training run (base config, C=64 T=128) ------------------
# runs/sc_v2/{best.pt,last.pt,history.json} 를 매 epoch 저장. --resume 은
# last.pt 에서 이어감(새 실행에도 안전: epoch 1부터). 목표 ~85%.
!{sys.executable} -m train.train --config configs/base.yaml --tag sc_v2 --resume \
    data.root={DATA_ROOT}

In [ ]:
# --- 8b. training from Python (8a 와 동일, 수정하기 쉬움) ----------------
from data.afe import AFEFrontend
from models.binary_matchboxnet import BinaryMatchboxNet
from train.train import Trainer, set_seed

cfg = load_config('configs/base.yaml', {'tag': 'sc_v2'})
cfg.data.root = DATA_ROOT

set_seed(cfg.train.seed)
afe = AFEFrontend(cfg.afe)
model = BinaryMatchboxNet(cfg.model)
print(model.describe())

train_loader, val_loader, test_loader = build_dataloaders(
    cfg.data, cfg.train.batch_size, cfg.afe.sample_rate, seed=cfg.train.seed)
afe.init_thresholds(next(iter(train_loader))[0])   # Cerutti IV-A

trainer = Trainer(cfg, model, afe=afe)
trainer.fit(train_loader, val_loader, resume=True)  # last.pt 있으면 이어감
test = trainer.evaluate(test_loader)
print(f"\ntest acc {test['acc']:.4f}  ({'MEETS' if test['acc']>=0.85 else 'below'} 85%)")

In [ ]:
# --- 8c. augmented training (증강만 ON, 나머지 baseline과 동일) ----------
# 증강 학습: baseline과 동일 조건(C64 T128, seed 1234, 100ep) + 증강만 ON
# CLAUDE.md 4: 파형 증강(time shift +-5 ms, noise)은 AFE 강건성에 특히 의미.
!{sys.executable} -m train.train --config configs/base.yaml --tag sc_v2_aug \
    data.aug_time_shift_ms=5 data.aug_noise_prob=0.3 \
    data.root={DATA_ROOT}

In [ ]:
# --- 9. (C, T) sweep -- 마일스톤: 85% 넘기는 최소 (C,T) ------------------
# 길다 (len(C)*len(T) runs). 처음엔 좁게 (예: --C 32 64 --T 64 128) 잡아 점당
# 시간 가늠. experiments/results/sweep.json 에 append 되고, 재실행 시 이미 있는
# (C,T) 점은 건너뛰어 point 단위로 resume 됨.
!{sys.executable} -m experiments.sweep --config configs/base.yaml \
    --C 16 32 48 64 --T 40 64 96 128 --epochs 100 \
    data.root={DATA_ROOT}

### 결과 git 커밋

이제 git 이 원격에 직접 있으니, `experiments/results/sweep.json` 을 원격 터미널
(또는 VSCode Source Control)에서 바로 커밋/push 하면 된다. 체크포인트(`runs/*.pt`)는
gitignore 되어 있으니 **push 하지 말 것**.
